# 第二章 编写 Prompt 的原则

  本章的主要内容为编写 Prompt 的原则，在本章中，我们将给出两个编写 Prompt 的原则与一些相关的策略，您可以练习编写高效的 Prompt，从而便捷而有效地使用 LLM。

<div class="toc">
    <ul class="toc-item">
        <li><span><a href="#一环境配置" data-toc-modified-id="一、环境配置">一、环境配置</a></span></li>
        <li>
            <span><a href="#二两个基本原则" data-toc-modified-id="二、两个基本原则">二、两个基本原则</a></span>
            <ul class="toc-item">
                <li><span><a href="#21-原则一编写清晰具体的指令" data-toc-modified-id="2.1 原则一：编写清晰、具体的指令">2.1 原则一：编写清晰、具体的指令</a></span></li>
                <li><span><a href="#22-给模型时间去思考" data-toc-modified-id="2.2 原则二：给模型时间去思考">2.2 原则二：给模型时间去思考</a></span></li>
            </ul>
        </li>
        <li><span><a href="#三局限性" data-toc-modified-id="三、局限性">三、局限性</a></span>
        </li>
    </ul>
</div>

## 一、环境配置

本教程使用 OpenAI 所开放的 ChatGPT API，因此您需要首先拥有一个 ChatGPT 的 API_KEY（也可以直接访问官方网址在线测试），然后需要安装 OpenAI 的第三方库。为了兼顾简便与兼容性，本教程将介绍在 ```Python 3``` 环境中基于 ```openai.api_key``` 方法的配置。另有基于环境变量的配置方法，详情请参考 [OpenAI 官方文档](https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety)。

首先需要安装 OpenAI 库：
```bash
pip install openai
```

In [10]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown

loaded = load_dotenv(find_dotenv(), override=True)
# 从环境变量中获取 OpenAI API Key 或者直接赋值
API_KEY = os.getenv("API_KEY")

# 如果您使用的是官方 API，就直接用 https://api.siliconflow.cn/v1 就行。
BASE_URL = "https://api.siliconflow.cn/v1"

In [12]:
# 实例化 OpenAI 对象
# 传入参数：OpenAI API Key（必需）、Base URL 和最大重试次数
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=3)

整个课程将以 gpt-3.5-turbo 模型为例。我们将在后续课程中深入探究 OpenAI 提供的 [Chat Completions API](https://platform.openai.com/docs/guides/gpt/chat-completions-api) 的使用方法，在此处，我们先将它封装成一个函数，您无需知道其内部机理，仅需知道调用该函数，以 Prompt 为输入参数，其将会输出对应的 Completion （回答结果）即可。

In [16]:
# 参数 n，整数或 Null，可选项，默认为 1。为每条输入信息生成多少个聊天完成选项。
# 参数 temperature，实数值或 Null，可选项，默认为 1。使用的采样温度，介于 0 和 2 之间。0.8 等较高值会使输出更加随机，而 0.2 等较低值会使输出更加集中和确定。

def get_completions(llm_prompt, model_endpoint):
    extra_body = {}
    if "Qwen3" in model_endpoint:
        extra_body={
            "enable_thinking": False
        }
        
    response = client.chat.completions.create(model=model_endpoint,
                                              messages=[
                                                        {"role": "user",
                                                         "content": llm_prompt
                                                        }
                                                       ],
                                              n=1, temperature=0, seed=42,
                                              presence_penalty=0, frequency_penalty=0,
                                              max_tokens=512, extra_body = extra_body
                                             )

    return response.choices[0].message.content.strip()

## 二、两个基本原则

### 2.1 原则一：编写清晰、具体的指令

您应该通过提供尽可能清晰和具体的指令来表达您希望模型执行的操作。这将引导模型给出正确的输出，并降低您得到无关或不正确响应的可能性。清晰的指令不意味着必须简短，在许多情况下，更长的 Prompt 实际上更清晰，且提供了更多上下文，也就可能产生更详细更相关的输出。

**2.1.1 使用分隔符清晰地表示输入的不同部分**

分隔符可以是：```，""，<>，:，\<tag> \</tag>等。

您可以使用任何明显的标点符号将特定的文本部分与 Prompt 的其余部分分开。标记的形式不限，只需要让模型明确知道这是一个单独部分。使用分隔符可以有效避免提示词注入( Prompt injection )。提示词注入是指如果允许用户将某些输入添加到（开发者预定义的） Prompt 中，则所提供的指令可能会与开发者想要执行的操作相冲突，从而使 LLM 遵循用户输入的指令，而非执行开发者预期的操作。即，输入里面可能包含其他指令，会覆盖掉您的指令。对此，使用分隔符是一个不错的策略。

在以下的例子中，我们给出一段话并要求 GPT 进行总结，在该示例中我们使用 ``` 来作为分隔符。


In [ ]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

# 中文版见下一个 cell
text = f"""
You should express what you want a model to do by \
providing instructions that are as clear and \
specific as you can possibly make them. \
This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant \
or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. \
In many cases, longer prompts provide more clarity \
and context for the model, which can lead to \
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```{text}```
"""
response = get_completions(prompt, llm)
print(response)

Clear and specific instructions should be provided to guide a model towards the desired output, and longer prompts can provide more clarity and context for the model, leading to more detailed and relevant outputs.

In [ ]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

text = f"""
您应该提供尽可能清晰、具体的指示，以表达您希望模型执行的任务。\
这将引导模型朝向所需的输出，并降低收到无关或不正确响应的可能性。\
不要将写清晰的提示词与写简短的提示词混淆。\
在许多情况下，更长的提示词可以为模型提供更多的清晰度和上下文信息，从而导致更详细和相关的输出。
"""
# 需要总结的文本内容
prompt = f"""
把用三个反引号括起来的文本总结成一句话。
```{text}```
"""
# 指令内容，使用 ``` 来分隔指令和待总结的内容
response = get_completions(prompt, llm)
print(response)


提供清晰具体的指示，避免无关或不正确响应，不要混淆写清晰和写简短，更长的提示可以提供更多清晰度和上下文信息，导致更详细和相关的输出。

**2.1.2 寻求结构化的输出**

输出可以是 Json、HTML 等格式。

第二个策略是要求生成一个结构化的输出，这可以使模型的输出更容易被我们解析，例如，您可以在 Python 中将其读入字典或列表中。

在以下示例中，我们要求 GPT 生成三本书的标题、作者和类别，并要求 GPT 以 Json 的格式返回给我们，为便于解析，我们指定了 Json 的键。

In [22]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
Generate a list of three made-up book titles along \ 
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completions(prompt, llm)
print(response)


```json
[
  {
    "book_id": 1,
    "title": "The Clockwork Paradox",
    "author": "Evelyn Thorne",
    "genre": "Science Fiction"
  },
  {
    "book_id": 2,
    "title": "Whispers of the Forgotten Grove",
    "author": "Lila Morn",
    "genre": "Fantasy"
  },
  {
    "book_id": 3,
    "title": "Ashes and Algorithms",
    "author": "Dr. Orion Vex",
    "genre": "Thriller"
  }
]
```


In [23]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
请生成包括书名、作者和类别的三本虚构书籍清单，\
并以 JSON 格式提供，其中包含以下键:book_id、title、author、genre。
"""
response = get_completions(prompt, llm)
print(response)


```json
[
  {
    "book_id": 1,
    "title": "星尘回声",
    "author": "林墨",
    "genre": "科幻"
  },
  {
    "book_id": 2,
    "title": "时光褶皱",
    "author": "苏晚晴",
    "genre": "奇幻"
  },
  {
    "book_id": 3,
    "title": "暗河之语",
    "author": "陈深",
    "genre": "悬疑"
  }
]
```


**2.1.3 要求模型检查是否满足条件**

如果任务包含不一定能满足的假设（条件），我们可以告诉模型先检查这些假设，如果不满足，则会指出并停止执行后续的完整流程。您还可以考虑可能出现的边缘情况及模型的应对，以避免意外的结果或错误发生。

在如下示例中，我们将分别给模型两段文本，分别是制作茶的步骤以及一段没有明确步骤的文本。我们将要求模型判断其是否包含一系列指令，如果包含则按照给定格式重新编写指令，不包含则回答“未提供步骤”。

In [25]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

text_1 = f"""
Making a cup of tea is easy! First, you need to get some \ 
water boiling. While that's happening, \ 
grab a cup and put a tea bag in it. Once the water is \ 
hot enough, just pour it over the tea bag. \ 
Let it sit for a bit so the tea can steep. After a \ 
few minutes, take out the tea bag. If you \ 
like, you can add some sugar or milk to taste. \ 
And that's it! You've got yourself a delicious \ 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completions(prompt, llm)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.  
Step 2 - Grab a cup and put a tea bag in it.  
Step 3 - Once the water is hot enough, pour it over the tea bag.  
Step 4 - Let it sit for a bit so the tea can steep.  
Step 5 - After a few minutes, take out the tea bag.  
Step 6 - If you like, add some sugar or milk to taste.  
Step 7 - And that's it! You've got yourself a delicious cup of tea to enjoy.


In [27]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \ 
walk in the park. The flowers are blooming, and the \ 
trees are swaying gently in the breeze. People \ 
are out and about, enjoying the lovely weather. \ 
Some are having picnics, while others are playing \ 
games or simply relaxing on the grass. It's a \ 
perfect day to spend time outdoors and appreciate the \ 
beauty of nature.
"""
prompt = f"""You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:
Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completions(prompt, llm)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


In [30]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

# 满足条件的输入（text中提供了步骤）
text_1 = f"""
泡一杯茶很容易。首先，需要把水烧开。\
在等待期间，拿一个杯子并把茶包放进去。\
一旦水足够热，就把它倒在茶包上。\
等待一会儿，让茶叶浸泡。几分钟后，取出茶包。\
如果您愿意，可以加一些糖或牛奶调味。\
就这样，您可以享受一杯美味的茶了。
"""
prompt = f"""
您将获得由三个引号括起来的文本。\
如果它包含一系列的指令，则需要按照以下格式重新编写这些指令：

第一步 - ...
第二步 - …
…
第N步 - …

如果文本中不包含一系列的指令，则直接写“未提供步骤”。"
\"\"\"{text_1}\"\"\"
"""
response = get_completions(prompt, llm)
print("Text 1 的总结:")
print(response)

Text 1 的总结:
第一步 - 把水烧开。  
第二步 - 拿一个杯子并把茶包放进去。  
第三步 - 一旦水足够热，就把它倒在茶包上。  
第四步 - 等待一会儿，让茶叶浸泡。  
第五步 - 几分钟后，取出茶包。  
第六步 - 如果您愿意，可以加一些糖或牛奶调味。  
第七步 - 这样，您可以享受一杯美味的茶了。


In [31]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

# 不满足条件的输入（text中未提供预期指令）
text_2 = f"""
今天阳光明媚，鸟儿在歌唱。\
这是一个去公园散步的美好日子。\
鲜花盛开，树枝在微风中轻轻摇曳。\
人们外出享受着这美好的天气，有些人在野餐，有些人在玩游戏或者在草地上放松。\
这是一个完美的日子，可以在户外度过并欣赏大自然的美景。
"""
prompt = f"""
您将获得由三个引号括起来的文本。\
如果它包含一系列的指令，则需要按照以下格式重新编写这些指令：

第一步 - ...
第二步 - …
…
第N步 - …

如果文本中不包含一系列的指令，则直接写“未提供步骤”。"
\"\"\"{text_2}\"\"\"
"""
response = get_completions(prompt, llm)
print("Text 2 的总结:")
print(response)

Text 2 的总结:
未提供步骤


**2.1.4 提供少量示例**（少样本提示词，Few-shot prompting）

即在要求模型执行实际任务之前，提供给它少量成功执行任务的示例。

例如，在以下的示例中，我们告诉模型其任务是以一致的风格回答问题，并先给它一个孩子和祖父之间的对话的例子。孩子说，“请教我何为耐心”，祖父用下述风格的隐喻来回答。由于我们已经告诉模型要以一致的语气回答，因此现在我们问“请教我何为韧性”，由于模型已经有了这个少样本示例( few-shot example )，它将以类似的语气回答下一个任务。

In [32]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \ 
valley flows from a modest spring; the \ 
grandest symphony originates from a single note; \ 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completions(prompt, llm)
print(response)

<grandparent>: The oak tree that stands tall in the storm is not born from a single gust of wind, but from the roots that hold firm through years of drought and darkness. The flame that burns brightest is not kindled in a moment, but nurtured through countless trials, growing stronger with each challenge it faces. The mountain that touches the sky was once a grain of sand, and it rose not by force, but by enduring the weight of time.


In [33]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
您的任务是以一致的风格回答问题。

<孩子>: 教我耐心。

<祖父母>: 挖出最深峡谷的河流源于一处不起眼的泉眼；最宏伟的交响乐从单一的音符开始；最复杂的挂毯以一根孤独的线开始编织。

<孩子>: 教我韧性。
"""
response = get_completions(prompt, llm)
print(response)

<祖父母>: 沙漠中的仙人掌在烈日下扎根，风雨中挺立；峭壁上的藤蔓不靠沃土，却攀向云端；夜航的星辰不因黑暗而熄灭，只因心中有光。


### 2.2 给模型时间去思考

如果您发现模型推理过程过于匆忙，导致得出了错误的结论，那么您应该尝试重新构思 Prompt ，要求模型在提供最终答案之前开展**思维链**，或进行一系列相关推理（a chain or series of relevant reasoning）。换句话说，如果您给模型一个在短时间内或用少量文字无法完成的复杂任务，它的输出结果就容易出错。这种情况对人来说也是类似：如果您要求某人完成复杂的数学问题，又不给足够时间计算出答案，他们也可能会犯错误。因此，在这些情况下，您应该指示模型花更多时间思考问题，让它在任务上花费更多计算资源。

**2.2.1 指定完成任务所需的步骤**

接下来我们将通过给定一个复杂任务，给出完成该任务的一系列步骤，来展示这一策略的效果。

首先我们描述了杰克和吉尔的故事，并给出提示词执行以下操作：首先，用一句话概括三个反引号限定的文本。第二，将摘要翻译成法语。第三，在法语摘要中列出每个名称。第四，输出包含以下键的 JSON 对象：法语摘要和人名个数。要求输出以换行符分隔。

In [34]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

text = f"""
In a charming village, siblings Jack and Jill set out on \ 
a quest to fetch water from a hilltop \ 
well. As they climbed, singing joyfully, misfortune \ 
struck—Jack tripped on a stone and tumbled \ 
down the hill, with Jill following suit. \ 
Though slightly battered, the pair returned home to \ 
comforting embraces. Despite the mishap, \ 
their adventurous spirits remained undimmed, and they \ 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completions(prompt_1, llm)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
The text tells the story of siblings Jack and Jill who go on an adventure to fetch water from a hilltop well, face a mishap when Jack falls, but continue their exploration with joy.  
La texte raconte l'histoire de deux frères et sœurs, Jack et Jill, qui entreprennent une aventure pour ramasser de l'eau dans une source située au sommet d'une colline, subissent un accident lorsque Jack trébuche, mais continuent leur exploration avec joie.  
Liste des noms dans le résumé français :  
- Jack  
- Jill  

{
  "french_summary": "La texte raconte l'histoire de deux frères et sœurs, Jack et Jill, qui entreprennent une aventure pour ramasser de l'eau dans une source située au sommet d'une colline, subissent un accident lorsque Jack trébuche, mais continuent leur exploration avec joie.",
  "num_names": 2
}


In [35]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

text = f"""
在一个迷人的村庄里，兄妹杰克和吉尔出发去一个山顶井里打水。\
他们一边唱着欢乐的歌，一边往上爬，\
然而不幸降临——杰克绊了一块石头，从山上滚了下来，吉尔紧随其后。\
虽然略有些摔伤，但他们还是回到了温馨的家中。\
尽管出了这样的意外，他们的冒险精神依然没有减弱，继续充满愉悦地探索。
"""
# example 1
prompt_1 = f"""
执行以下操作：
1-用一句话概括下面用三个反引号括起来的文本。
2-将摘要翻译成法语。
3-在法语摘要中列出每个人名。
4-输出一个 JSON 对象，其中包含以下键：French_summary，num_names。

请用换行符分隔您的答案。

Text:
```{text}```
"""
response = get_completions(prompt_1, llm)
print("prompt 1:")
print(response)

prompt 1:
兄妹杰克和吉尔去山顶井打水，途中发生意外，但最终安全回家并继续冒险。

French_summary :  
Les frères et sœurs Jack et Gill partent chercher de l'eau dans une pompe située en haut d'une montagne. En chemin, un accident survient : Jack trébuche sur une pierre et roule en bas de la montagne, suivi par Gill. Bien qu'ils aient eu quelques bosses, ils rentrent en sécurité chez eux. Malgré cet incident, leur esprit d'aventure reste intact et ils continuent d'explorer avec joie.

num_names : 2

```json
{
  "French_summary": "Les frères et sœurs Jack et Gill partent chercher de l'eau dans une pompe située en haut d'une montagne. En chemin, un accident survient : Jack trébuche sur une pierre et roule en bas de la montagne, suivi par Gill. Bien qu'ils aient eu quelques bosses, ils rentrent en sécurité chez eux. Malgré cet incident, leur esprit d'aventure reste intact et ils continuent d'explorer avec joie.",
  "num_names": 2
}
```


上述输出仍然存在一定问题，例如，键“姓名”会被替换为法语（译注：在英文原版中，对应指令第三步的输出为 'Noms:',为Name的法语，这种行为难以预测，并可能为导出带来困难）

因此，我们将Prompt加以改进，该 Prompt 前半部分不变，同时**确切指定了输出的格式**。

In [36]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in French summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completions(prompt_2, llm)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Text: <在一个迷人的村庄里，兄妹杰克和吉尔出发去一个山顶井里打水。他们一边唱着欢乐的歌，一边往上爬，然而不幸降临——杰克绊了一块石头，从山上滚了下来，吉尔紧随其后。虽然略有些摔伤，但他们还是回到了温馨的家中。尽管出了这样的意外，他们的冒险精神依然没有减弱，继续充满愉悦地探索。>
Summary: <兄妹杰克和吉尔去山顶井打水，途中发生意外但最终安全回家。>
Translation: <Les frères et sœurs Jack et Gill vont chercher de l'eau dans une puits au sommet d'une montagne, mais un accident survient en chemin et ils rentrent enfin en sécurité à la maison.>
Names: <["Jack", "Gill"]>
Output JSON: {"french_summary": "Les frères et sœurs Jack et Gill vont chercher de l'eau dans une puits au sommet d'une montagne, mais un accident survient en chemin et ils rentrent enfin en sécurité à la maison.", "num_names": 2}


In [37]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt_2 = f"""
1-用一句话概括下面用<>括起来的文本。
2-将摘要翻译成英语。
3-在英语摘要中列出每个名称。
4-输出一个 JSON 对象，其中包含以下键：English_summary，num_names。

请使用以下格式：
文本：<要总结的文本>
摘要：<摘要>
翻译：<摘要的翻译>
名称：<英语摘要中的名称列表>
输出 JSON：<带有 English_summary 和 num_names 的 JSON>

Text: <{text}>
"""
response = get_completions(prompt_2, llm)
print("\nprompt 2:")
print(response)


prompt 2:
摘要：<杰克和吉尔在村庄中前往山顶井打水，途中发生意外，但最终安全回家并继续冒险。>  
翻译：<Jack and Jill went to a mountain well in a charming village, had an accident on the way, but eventually returned home safely and continued their adventures.>  
名称：<["Jack", "Jill"]>  
输出 JSON：<{"English_summary": "Jack and Jill went to a mountain well in a charming village, had an accident on the way, but eventually returned home safely and continued their adventures.", "num_names": 2}>


**2.2.2 指导模型在下结论之前找出一个自己的解法**

明确地指引模型在匆匆做决策之前，要自己思考出一份解决方案。有时这样会得到更好的结果。这与之前所述思想类似，即给模型时间思考。

接下来我们会给出一个问题和一份来自学生的解答，要求模型判断解答是否正确：

In [38]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \ 
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completions(prompt, llm)
print(response)

Let's analyze the **student's solution** step by step to determine if it is correct.

---

### **Given:**
- **Land cost:** $100 per square foot
- **Solar panel cost:** $250 per square foot
- **Maintenance cost:** $100,000 per year **plus** $10 per square foot

---

### **Let x be the size of the installation in square feet.**

#### **1. Land cost:**  
- Correct: $100x$ (since it's $100 per square foot)

#### **2. Solar panel cost:**  
- Correct: $250x$ (since it's $250 per square foot)

#### **3. Maintenance cost:**  
- **Incorrect:** The student wrote **100,000 + 100x**, but the correct expression is **100,000 + 10x**.  
  - The maintenance cost is a **flat $100,000 per year** **plus** an **additional $10 per square foot**.

---

### **Total cost:**
- The student added:  
  $100x + 250x + 100,000 + 100x = 450x + 100,000$  
- But the correct total cost should be:  
  $100x + 250x + 100,000 + 10x = 360x + 100,000$

---

### ✅ **Conclusion:**
The **student's solution is incorrect**.

###

In [39]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
判断学生的解决方案是否正确。

问题:
我正在建造一个太阳能发电站，需要帮助计算财务。

    土地费用为 100美元/平方英尺
    我可以以 250美元/平方英尺的价格购买太阳能电池板
    我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元
    作为平方英尺数的函数，首年运营的总费用是多少。

学生的解决方案：
设x为发电站的大小，单位为平方英尺。
费用：

    土地费用：100x
    太阳能电池板费用：250x
    维护费用：100,000美元+100x
    总费用：100x+250x+100,000美元+100x=450x+100,000美元
"""
response = get_completions(prompt, llm)
print(response)

学生的解决方案 **基本正确**，但存在一些 **小错误**，特别是在 **维护费用** 的计算部分。我们来逐步分析：

---

### ✅ 问题回顾：

- 土地费用：100美元/平方英尺
- 太阳能电池板费用：250美元/平方英尺
- 维护费用：每年固定支付10万美元，**加上** 每平方英尺10美元
- 要求：作为平方英尺数的函数，**首年运营的总费用** 是多少。

---

### ✅ 学生的解答：

设 $ x $ 为发电站的大小（单位：平方英尺）。

- 土地费用：$ 100x $
- 太阳能电池板费用：$ 250x $
- 维护费用：$ 100,000 + 10x $
- 总费用：$ 100x + 250x + 100,000 + 10x = 450x + 100,000 $

---

### 🔍 错误分析：

1. **维护费用部分**：
   - 学生写的是：**100,000美元 + 100x**
   - 但根据题目，维护费用是 **每年固定支付10万美元，加上每平方英尺10美元**。
   - 所以正确的表达应该是：**100,000 + 10x**

2. **总费用计算**：
   - 学生将维护费用错误地写为 **100,000 + 100x**，导致总费用计算错误。
   - 正确的总费用应为：
     $$
     100x + 250x + 100,000 + 10x = 450x + 100,000
     $$

---

### ✅ 正确的总费用函数：

$$
\text{总费用} = 100x + 250x + 100,000 + 10x = 450x + 100,000
$$

---

### ✅ 结论：

学生的解决方案


但是注意，学生的解决方案实际上是错误的。（*维护费用项100x应为10x，总费用450x应为360x*）

我们可以通过指导模型先自行找出一个解法来解决这个问题。

在接下来这个 Prompt 中，我们要求模型先自行解决这个问题，再根据自己的解法与学生的解法进行对比，从而判断学生的解法是否正确。同时，我们给定了输出的格式要求。通过拆分任务、明确步骤，让模型有更多时间思考，有时可以获得更准确的结果。在这个例子中，学生的答案是错误的，但如果我们没有先让模型自己计算，那么可能会被误导以为学生是正确的。

In [40]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem. 
- Then compare your solution to the student's solution \ 
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completions(prompt, llm)
print(response)

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
```

Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```

Actual solution:
```
Let x be the size of the installation in square feet.

1. Land cost: $100 per square foot → 100x
2. Solar panel cost: $250 per square foot → 250x
3. Maintenance cost: $100,000 flat fee + $10 per square foot → 100,000 + 10x

Total cost = Land cost + Solar panel cost + Maintenance cost  
= 100x + 250x + 100,000 + 10x  
= (100 + 250 + 10)x 

In [41]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
请判断学生的解决方案是否正确，请通过如下步骤解决这个问题：

步骤：
首先，自己解决问题，解决问题时列数学表达式。
然后将您的解决方案与学生的解决方案进行比较，并评估学生的解决方案是否正确。
在自己完成问题之前，请勿决定学生的解决方案是否正确。

使用以下格式：

问题：问题文本
学生的解决方案：学生的解决方案文本
实际解决方案和步骤：实际解决方案和步骤文本
**学生的计算结果：学生的计算结果文本
实际计算结果：实际计算结果文本
学生的计算结果和实际计算结果是否相同：是或否
学生的解决方案和实际解决方案是否相同：是或否**
学生的成绩：正确或不正确

问题：
我正在建造一个太阳能发电站，需要帮助计算财务。
- 土地费用为每平方英尺100美元
- 我可以以每平方英尺250美元的价格购买太阳能电池板
- 我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元
作为平方英尺数的函数，首年运营的总费用是多少。

学生的解决方案：

设x为发电站的大小，单位为平方英尺。
费用：
1. 土地费用：100x
2. 太阳能电池板费用：250x
3. 维护费用：100,000+100x
总费用：100x+250x+100,000+100x=450x+100,000

实际解决方案和步骤：
"""
response = get_completions(prompt, llm)
print(response)

问题：  
我正在建造一个太阳能发电站，需要帮助计算财务。  
- 土地费用为每平方英尺100美元  
- 我可以以每平方英尺250美元的价格购买太阳能电池板  
- 我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元  
作为平方英尺数的函数，首年运营的总费用是多少。

学生的解决方案：  
设x为发电站的大小，单位为平方英尺。  
费用：  
1. 土地费用：100x  
2. 太阳能电池板费用：250x  
3. 维护费用：100,000+100x  
总费用：100x+250x+100,000+100x=450x+100,000

实际解决方案和步骤：  
设x为发电站的大小（单位：平方英尺）。  
- 土地费用：100美元/平方英尺 × x = 100x  
- 太阳能电池板费用：250美元/平方英尺 × x = 250x  
- 维护费用：固定10万美元 + 每平方英尺10美元 × x = 100,000 + 10x  
总费用 = 土地费用 + 太阳能电池板费用 + 维护费用  
= 100x + 250x + 100,000 + 10x  
= (100 + 250 + 10)x + 100,000  
= 360x + 100,000

**学生的计算结果：** 450x + 100,000  
**实际计算结果：** 360x + 100,000  
**学生的计算结果和实际计算结果是否相同：** 否  
**学生的解决方案和实际解决方案是否相同：** 否  
**学生的成绩：** 不正确


## 三、局限性

**开发大模型相关应用时请务必铭记：**


**虚假知识**：模型偶尔会生成一些看似真实实则编造的知识

虽然模型在训练过程中接触了大量的知识，但它并没有*完全*记住所见的信息，因此它不甚清楚自己知识的边界。这意味着它可能会尝试回答主题晦涩难懂的问题，并编造听起来合理但实际上并不正确的答案。我们称这些编造的想法为幻觉（Hallucination）。

如下示例展示了大模型的幻觉。我们要求告诉我们 Boie 公司生产的 *AeroGlide UltraSlim Smart Toothbrush* 产品的信息，事实上，这个公司是真实存在的，但产品是编造的，而模型一本正经地提供了它编造的知识，而且迷惑性很强。



In [42]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completions(prompt, llm)
print(response)

The **AeroGlide UltraSlim Smart Toothbrush by Boie** is a modern, sleek, and technologically advanced toothbrush designed to combine oral health benefits with smart features. Boie is a well-known brand in the personal care and wellness space, particularly for its innovative and minimalist approach to products like toothbrushes, toothpaste, and other oral hygiene tools.

### Key Features of the AeroGlide UltraSlim Smart Toothbrush:

1. **Ultra-Slim Design**:
   - The toothbrush has a very narrow, ergonomic handle that is designed to fit comfortably in the hand and reach tight spaces in the mouth more easily.
   - Its slim profile is ideal for people with smaller mouths or those who prefer a more compact brush.

2. **Smart Technology**:
   - The brush includes **Bluetooth connectivity** and a **smart app** (available on iOS and Android) that allows users to track their brushing habits, such as duration, pressure, and coverage.
   - It provides real-time feedback through the app, helping 

In [43]:
# 我们使用 Qwen/Qwen3-8B
llm = "Qwen/Qwen3-8B"

prompt = f"""
告诉我 Boie 公司生产的 AeroGlide UltraSlim Smart Toothbrush 的相关信息
"""
response = get_completions(prompt, llm)
print(response)

关于 **Boie 公司生产的 AeroGlide UltraSlim Smart Toothbrush**，目前公开信息中并没有明确的、广泛认可的公司名为“Boie”生产这款产品。因此，可能存在以下几种情况：

---

### 1. **可能的拼写错误或混淆**
- “Boie” 可能是 “Boe” 或 “Boie” 的误写。
- “AeroGlide UltraSlim Smart Toothbrush” 可能是某款智能牙刷的名称，但目前没有明确的资料显示该产品由 Boie 公司生产。

---

### 2. **可能的公司与产品混淆**
- 有些公司可能会使用类似名称的商标或产品线，例如：
  - **AeroGlide** 是 **Oral-B**（欧乐B）旗下的一款牙刷品牌，但通常指的是 **AeroGlide** 系列的普通牙刷，而不是“UltraSlim Smart”版本。
  - **UltraSlim Smart Toothbrush** 可能是 **Colgate**（高露洁）或其他品牌的产品，但没有明确的证据表明它由 Boie 公司生产。

---

### 3. **可能的虚构或非主流产品**
- 如果你是在某个特定平台（如电商平台、社交媒体、论坛）看到这个产品名称，它可能是某个小众品牌、定制产品，或者是虚构的、营销用的名称。
- 也有可能是用户将多个品牌或产品名称混淆了。

---

### 4. **建议的验证方法**
如果你想要确认这款牙刷的信息，可以尝试以下方法：

- **查看产品包装或说明书**：通常会印有品牌名称、型号、生产厂商等信息。
- **搜索产品名称**：在 Google、Amazon、京东等平台搜索 “AeroGlide UltraSlim Smart Toothbrush”。
- **查看品牌官网**：访问 Boie、Oral-B、Colgate 等品牌的官方网站，确认是否有该产品。
- **联系客服或销售商**：如果你是从某个商家购买的，可以联系他们确认产品来源和品牌信息。

---

### 5. **可能的替代产品**
如果你是在寻找一款 **智能牙刷**，以下是一些知名品牌的产品，供你参考：

- **Oral-B Genius Smart Toothbrush**（欧乐B智齿牙刷）
- **Colgat

由于很容易以假乱真，请读者根据在本系列教程中所学知识，在构建自己的应用程序时尽量避免幻觉情况。幻觉是大模型的一个已知缺陷（注：截至2023年7月），OpenAI也在努力解决该问题。

在您希望模型根据文本生成回答时，另一种减少幻觉的策略是先要求模型获取来源于该文本的所有引用信息（任何相关引用，any relevant quotes），然后要求它基于所引用的信息来回答问题，这使得我们能根据答案追溯源文档，通常对减少幻觉非常有帮助。

**关于反斜杠使用的说明：**

在本教程中，我们使用反斜杠 \ 来使文本适应屏幕大小以提高阅读体验，而没有用换行符 \n 。GPT-3 并不受换行符（newline characters）的影响，但在您调用其他大模型时，需额外考虑换行符是否会影响模型性能。